# 13. POMDP와 Belief-Space Planning

POMDP는 상태를 직접 볼 수 없고 관측으로 belief를 업데이트하며 행동을 선택하는 모델이다.

$$b'(s')=\eta p(z\mid s')\sum_s p(s'\mid s,a)b(s)$$

MDP가 상태 $s$에서 planning한다면, POMDP는 belief $b$에서 planning한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Tiger Problem 미니 POMDP

문 뒤에 tiger가 있고, 로봇은 listen 또는 문 열기를 선택한다. Listen은 비용이 있지만 belief를 날카롭게 만든다.

In [ ]:
states=['TL','TR']  # tiger left/right
actions=['listen','open_left','open_right']
p_correct=0.85
gamma=0.95

def update_belief(b, obs):
    # b = P(TL)
    if obs=='hear_left':
        like_tl=p_correct; like_tr=1-p_correct
    else:
        like_tl=1-p_correct; like_tr=p_correct
    num=like_tl*b; den=num+like_tr*(1-b)
    return num/den

def reward(b,a):
    if a=='listen': return -1
    if a=='open_left': return b*(-100)+(1-b)*10
    if a=='open_right': return b*10+(1-b)*(-100)

belief_grid=np.linspace(0.001,0.999,401)
V=np.maximum([reward(b,'open_left') for b in belief_grid],[reward(b,'open_right') for b in belief_grid])
policy=np.array(['open_left' if reward(b,'open_left')>reward(b,'open_right') else 'open_right' for b in belief_grid],dtype=object)
for _ in range(80):
    Vnew=np.zeros_like(V); pol=[]
    for i,b in enumerate(belief_grid):
        q_open_l=reward(b,'open_left')
        q_open_r=reward(b,'open_right')
        # expected value of listen over observations
        p_hl=p_correct*b+(1-p_correct)*(1-b)
        b_hl=update_belief(b,'hear_left'); b_hr=update_belief(b,'hear_right')
        v_hl=np.interp(b_hl,belief_grid,V); v_hr=np.interp(b_hr,belief_grid,V)
        q_listen=-1+gamma*(p_hl*v_hl+(1-p_hl)*v_hr)
        qs=[q_listen,q_open_l,q_open_r]; ai=int(np.argmax(qs))
        Vnew[i]=qs[ai]; pol.append(actions[ai])
    if np.max(np.abs(Vnew-V))<1e-5: break
    V=Vnew; policy=np.array(pol,dtype=object)

fig,axes=plt.subplots(2,1,figsize=(9,7),sharex=True)
axes[0].plot(belief_grid,V,color='#534AB7',lw=2.5); axes[0].set_ylabel('V(b)'); axes[0].grid(alpha=0.25)
colors={'listen':'#1D9E75','open_left':'#E85D24','open_right':'#534AB7'}
for a in actions:
    mask=policy==a
    axes[1].scatter(belief_grid[mask],np.zeros(mask.sum()),s=8,color=colors[a],label=a)
axes[1].set_yticks([]); axes[1].set_xlabel('belief P(tiger left)'); axes[1].legend(); axes[1].set_title('optimal action in belief space')
plt.tight_layout(); plt.savefig('assets/13_tiger_pomdp.png',dpi=150,bbox_inches='tight'); plt.show()
for b in [0.1,0.5,0.9]:
    idx=np.argmin(abs(belief_grid-b)); print('belief',b,'->',policy[idx])

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| POMDP | partially observable decision process | Ch.15 POMDP |
| Belief state | 확률분포를 planning state로 사용 | localization + planning 결합 |
| Information gathering | listen처럼 정보를 얻는 행동 | active perception의 기본 |